In [32]:
import pyarrow
import pandas as pd
import numpy as np
from pathlib import Path


EXTRAÇÃO: importação dos dados

In [33]:
customer = pd.read_csv(Path("C:\\Users\\Divini\\Documents\\projetos\\projeto etl1\\data\\customer.csv"))
print(f"total de registros: {len(customer)}")
print(f"total colunas: {len(customer.columns)}")
customer.head()

total de registros: 5050
total colunas: 7


,Customer ID,Age,Gender,Subscription Status,Previous Purchases,Frequency of Purchases,Location
0,2701,22,Female,No,36.0,Weekly,California
1,521,51,Male,Yes,20.0,Quarterly,South Carolina
2,3157,18,Female,NaN,18.0,Monthly,Montana
3,1687,22,Male,No,25.0,Annually,Illinois
4,2929,40,Female,No,17.0,Weekly,Alabama


In [34]:
item = pd.read_csv(Path("C:\\Users\\Divini\\Documents\\projetos\\projeto etl1\\data\\item.csv"))
print(f"total de registros: {len(item)}")
print(f"total colunas: {len(item.columns)}")
item.head()

total de registros: 5050
total colunas: 11


,Item Purchased,Size,Color,Category,Season,Review Rating,Purchase Amount (USD),Discount Applied,Shipping Type,Payment Method,Customer ID
0,T-shirt,XL,Olive,Clothing,Winter,3.2,68.0,No,Standard,Cash,2701
1,Sunglasses,M,White,Accessories,Spring,3.9,84.0,Yes,Free Shipping,Debit Card,521
2,Shirt,M,Black,Clothing,Winter,3.1,50.0,No,2-Day Shipping,Cash,3157
3,Gloves,L,Red,Accessories,Fall,4.2,75.0,No,Store Pickup,Cash,1687
4,NaN,L,Yellow,Accessories,Spring,3.6,80.0,No,Store Pickup,Credit Card,2929


DATA QUALITY E DETECÇÃO DE ANOMALIAS

In [35]:
item.info()
print("")
print("")
customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Item Purchased         5009 non-null   str    
 1   Size                   4680 non-null   str    
 2   Color                  5050 non-null   str    
 3   Category               5014 non-null   str    
 4   Season                 5027 non-null   str    
 5   Review Rating          4449 non-null   float64
 6   Purchase Amount (USD)  4494 non-null   float64
 7   Discount Applied       5050 non-null   str    
 8   Shipping Type          5032 non-null   str    
 9   Payment Method         5039 non-null   str    
 10  Customer ID            5050 non-null   int64  
dtypes: float64(2), int64(1), str(8)
memory usage: 671.8 KB


<class 'pandas.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------       

In [36]:
print("=== VALORES NULOS ===")
null_counts = customer.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Nenhum valor nulo encontrado")
print("")
print("")
print("=== VALORES NULOS ===")
null_counts = item.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Nenhum valor nulo encontrado")

=== VALORES NULOS ===
Subscription Status        33
Previous Purchases        548
Frequency of Purchases     10
dtype: int64


=== VALORES NULOS ===
Item Purchased            41
Size                     370
Category                  36
Season                    23
Review Rating            601
Purchase Amount (USD)    556
Shipping Type             18
Payment Method            11
dtype: int64


In [37]:
# 1 Detectar anomalias no DataFrame: Customer
total_customer = len(customer)

# Criando as condições para as anomalias
missing_subscription = customer['Subscription Status'].isna()
invalid_age = (customer['Age'] < 18) | (customer['Age'] > 120)
invalid_prev_purchases = customer['Previous Purchases'] < 0
duplicated_customers = customer.duplicated(subset=['Customer ID'], keep=False)

print("=== ANOMALIAS DETECTADAS (CUSTOMER) ===")
print(f"Status de assinatura ausente (NaN): {missing_subscription.sum():,} ({missing_subscription.sum()/total_customer*100:.2f}%)")
print(f"Idade inválida (<18 ou >120): {invalid_age.sum():,} ({invalid_age.sum()/total_customer*100:.2f}%)")
print(f"Compras anteriores negativas (<0): {invalid_prev_purchases.sum():,} ({invalid_prev_purchases.sum()/total_customer*100:.2f}%)")
print(f"IDs de cliente duplicados: {duplicated_customers.sum():,} ({duplicated_customers.sum()/total_customer*100:.2f}%)")



# 2 Detectar anomalias no DataFrame: item

total_item = len(item)

# Criando as máscaras (condições) para as anomalias
missing_item_name = item['Item Purchased'].isna()
invalid_review = (item['Review Rating'] < 1.0) | (item['Review Rating'] > 5.0)
invalid_purchase_amount = item['Purchase Amount (USD)'] <= 0

print("\n=== ANOMALIAS DETECTADAS (ITEM) ===")
print(f"Nome do item ausente (NaN): {missing_item_name.sum():,} ({missing_item_name.sum()/total_item*100:.2f}%)")
print(f"Avaliação inválida (<1.0 ou >5.0): {invalid_review.sum():,} ({invalid_review.sum()/total_item*100:.2f}%)")
print(f"Valor de compra inválido (<=0): {invalid_purchase_amount.sum():,} ({invalid_purchase_amount.sum()/total_item*100:.2f}%)")


=== ANOMALIAS DETECTADAS (CUSTOMER) ===
Status de assinatura ausente (NaN): 33 (0.65%)
Idade inválida (<18 ou >120): 0 (0.00%)
Compras anteriores negativas (<0): 0 (0.00%)
IDs de cliente duplicados: 100 (1.98%)

=== ANOMALIAS DETECTADAS (ITEM) ===
Nome do item ausente (NaN): 41 (0.81%)
Avaliação inválida (<1.0 ou >5.0): 0 (0.00%)
Valor de compra inválido (<=0): 0 (0.00%)


In [38]:
#tratamento para customer
print(f"Total de clientes antes da limpeza: {len(customer)}")
#Tratar valores nulos (NaN)
customer['Subscription Status'] = customer['Subscription Status'].fillna('Desconhecido')

#Ids duplicados
customer = customer.drop_duplicates(subset=['Customer ID'], keep='first')
print(f"Total de clientes após a limpeza: {len(customer)}\n")

#tratamento para item
print(f"Total de itens antes da limpeza: {len(item)}")

#Remover registros sem o nome do ite
item = item.dropna(subset=['Item Purchased'])
#Remover linhas duplicadas se houver
item = item.drop_duplicates()
print(f"Total de itens após a limpeza: {len(item)}")

Total de clientes antes da limpeza: 5050
Total de clientes após a limpeza: 5000

Total de itens antes da limpeza: 5050
Total de itens após a limpeza: 4960


TRANSFORMAÇÃO, AGREGAÇÃO E MÉTRICAS

In [39]:
# Criando uma nova coluna 'Faixa Etaria'
bins = [0, 18, 25, 35, 50, 120]
labels = ['Menor de 18', '18-25', '26-35', '36-50', 'Mais de 50']
customer['Faixa Etaria'] = pd.cut(customer['Age'], bins=bins, labels=labels)

# Converte "Yes" para 1 e "No" (ou outros) para 0
customer['Is_Subscriber'] = customer['Subscription Status'].apply(lambda x: 1 if x == 'Yes' else 0)

#Transformações na tabela item
taxa_cambio = 5.50
item['Purchase Amount (BRL)'] = item['Purchase Amount (USD)'] * taxa_cambio

#Ajuste de Strings e Padronização
item['Category'] = item['Category'].str.upper()
item['Season'] = item['Season'].str.capitalize()

#CRUZANDO TABELAS
#Gasto Total por Cliente 
# 1. Soma os gastos agrupando pelo ID do cliente
gasto_por_cliente = item.groupby('Customer ID')['Purchase Amount (USD)'].sum().reset_index()
# Renomeia a coluna para ficar claro
gasto_por_cliente.rename(columns={'Purchase Amount (USD)': 'Total Gasto Atual'}, inplace=True)

# 2. Junta essa informação na tabela de clientes
customer = pd.merge(customer, gasto_por_cliente, on='Customer ID', how='left')

# Preenche com 0 caso o cliente não tenha comprado nenhum item nessa extração
customer['Total Gasto Atual'] = customer['Total Gasto Atual'].fillna(0)

#Ticket Médio
# Conta quantos itens o cliente comprou nesta tabela de itens
itens_comprados = item.groupby('Customer ID').size().reset_index(name='Qtd Itens Comprados')
customer = pd.merge(customer, itens_comprados, on='Customer ID', how='left')
customer['Qtd Itens Comprados'] = customer['Qtd Itens Comprados'].fillna(0)

# Calcula o Ticket Médio das compras ATUAIS
customer['Ticket Medio Atual'] = customer['Total Gasto Atual'] / customer['Qtd Itens Comprados']

In [40]:
customer.head()


,Customer ID,Age,Gender,Subscription Status,Previous Purchases,Frequency of Purchases,Location,Faixa Etaria,Is_Subscriber,Total Gasto Atual,Qtd Itens Comprados,Ticket Medio Atual
0,2701,22,Female,No,36.0,Weekly,California,18-25,0,68.0,1.0,68.0
1,521,51,Male,Yes,20.0,Quarterly,South Carolina,Mais de 50,1,84.0,1.0,84.0
2,3157,18,Female,Desconhecido,18.0,Monthly,Montana,Menor de 18,0,50.0,1.0,50.0
3,1687,22,Male,No,25.0,Annually,Illinois,18-25,0,75.0,1.0,75.0
4,2929,40,Female,No,17.0,Weekly,Alabama,36-50,0,0.0,0.0,NaN


In [41]:
item.head()

,Item Purchased,Size,Color,Category,Season,Review Rating,Purchase Amount (USD),Discount Applied,Shipping Type,Payment Method,Customer ID,Purchase Amount (BRL)
0,T-shirt,XL,Olive,CLOTHING,Winter,3.2,68.0,No,Standard,Cash,2701,374.0
1,Sunglasses,M,White,ACCESSORIES,Spring,3.9,84.0,Yes,Free Shipping,Debit Card,521,462.0
2,Shirt,M,Black,CLOTHING,Winter,3.1,50.0,No,2-Day Shipping,Cash,3157,275.0
3,Gloves,L,Red,ACCESSORIES,Fall,4.2,75.0,No,Store Pickup,Cash,1687,412.5
5,Shorts,S,Beige,CLOTHING,Summer,3.4,41.0,No,2-Day Shipping,PayPal,3583,225.5


CARREGANDO EM PARQUET

In [43]:

output_dir = Path('C:\\Users\\Divini\\Documents\\projetos\\projeto etl1\\output data')
output_dir.mkdir(parents=True, exist_ok=True)


# SALVAR DATAFRAME: customer

customer = customer.drop(columns=['Subscription Status','Age'])


output_path_customer = output_dir / 'customer_clean.parquet'

# Exportar para Parquet
customer.to_parquet(output_path_customer, engine='pyarrow', index=False)

# Calcular tamanho do arquivo em MB
size_mb_customer = output_path_customer.stat().st_size / (1024 * 1024)

print("=== PARQUET: CUSTOMER ===")
print(f"Arquivo: {output_path_customer}")
print(f"Tamanho: {size_mb_customer:.3f} MB")



# SALVAR DATAFRAME: item
item = item.drop(columns=['Purchase Amount (USD)'])
output_path_item = output_dir / 'item_clean.parquet'

# Exportar para Parquet
item.to_parquet(output_path_item, engine='pyarrow', index=False)

# Calcular tamanho do arquivo em MB
size_mb_item = output_path_item.stat().st_size / (1024 * 1024)

print("\n=== PARQUET: ITEM ===")
print(f"Arquivo: {output_path_item}")
print(f"Tamanho: {size_mb_item:.3f} MB")

=== PARQUET: CUSTOMER ===
Arquivo: C:\Users\Divini\Documents\projetos\projeto etl1\output data\customer_clean.parquet
Tamanho: 0.065 MB

=== PARQUET: ITEM ===
Arquivo: C:\Users\Divini\Documents\projetos\projeto etl1\output data\item_clean.parquet
Tamanho: 0.062 MB


In [48]:
# VALIDAÇÃO: customer
df_customer_parquet = pd.read_parquet(output_path_customer)
print("=== VALIDAÇÃO: CUSTOMER ===")
print(f"Registros no Parquet: {len(df_customer_parquet):,}")
print(f"Registros esperados:  {len(customer):,}")
print(f"Match: {'OK' if len(df_customer_parquet) == len(customer) else 'ERRO'}")
print(f"\nColunas: {list(df_customer_parquet.columns)}")

# VALIDAÇÃO: item
df_item_parquet = pd.read_parquet(output_path_item)
print("\n=== VALIDAÇÃO: ITEM ===")
print(f"Registros no Parquet: {len(df_item_parquet):,}")
print(f"Registros esperados:  {len(item):,}")
print(f"Match: {'OK' if len(df_item_parquet) == len(item) else 'ERRO'}")
print(f"\nColunas: {list(df_item_parquet.columns)}")


=== VALIDAÇÃO: CUSTOMER ===
Registros no Parquet: 5,000
Registros esperados:  5,000
Match: OK

Colunas: ['Customer ID', 'Gender', 'Previous Purchases', 'Frequency of Purchases', 'Location', 'Faixa Etaria', 'Is_Subscriber', 'Total Gasto Atual', 'Qtd Itens Comprados', 'Ticket Medio Atual']

=== VALIDAÇÃO: ITEM ===
Registros no Parquet: 4,960
Registros esperados:  4,960
Match: OK

Colunas: ['Item Purchased', 'Size', 'Color', 'Category', 'Season', 'Review Rating', 'Discount Applied', 'Shipping Type', 'Payment Method', 'Customer ID', 'Purchase Amount (BRL)']
